# Hyperparameter-Optimierung: Random Forest & LSTM
**Wichtig:** Die Optimierung (Random Search) läuft *immer auf dem Server*.

Wir optimieren jeweils zwei **Deployment-Profile** pro Modell:
- **edge**: Suchräume und Settings sind für embedded/Edge begrenzt.
- **server**: Größere Suchräume für maximale Qualität.

Die Profile steuern also nur *den Ziel-Einsatz und den Parameter-Suchraum*,
nicht den Ort, an dem die Optimierung stattfindet.

- Zielvariable: `Group4-2_S6_VolumetricFlowRate`
- `lags`/`horizon` **nicht** Teil der Optimierung.
- Zeitspalten werden als Features ausgeschlossen.

In [ ]:

import os, json, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop, Nadam

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
print("TF version:", tf.__version__)

# ---- User-Settings ----
DATA_PATH = r"""C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_rate_limited.csv"""
TARGET_COL = "Group4-2_S6_VolumetricFlowRate"
EXCLUDE_COLS = ["time", "datetime", "recording_timestamp", "duration"]

# Zeitreihen-Parameter (nicht optimiert)
LAGS = 4
HORIZON = 4

# Gemeinsame Split-Einstellungen
TRAIN_FRACTION = 0.8
VAL_FRACTION = 0.2  # innerhalb des Trainings für LSTM genutzt


In [ ]:

import json
with open("/mnt/data/config_rf_edge_opt.json","r",encoding="utf-8") as f: print("RF EDGE:", json.load(f)["deployment_profile"], "| optimize_on:", json.load(open("/mnt/data/config_rf_edge_opt.json","r")).get("optimize_on"))
with open("/mnt/data/config_rf_server_opt.json","r",encoding="utf-8") as f: print("RF SERVER:", json.load(f)["deployment_profile"], "| optimize_on:", json.load(open("/mnt/data/config_rf_server_opt.json","r")).get("optimize_on"))
with open("/mnt/data/config_lstm_edge_opt.json","r",encoding="utf-8") as f: print("LSTM EDGE:", json.load(f)["deployment_profile"], "| optimize_on:", json.load(open("/mnt/data/config_lstm_edge_opt.json","r")).get("optimize_on"))
with open("/mnt/data/config_lstm_server_opt.json","r",encoding="utf-8") as f: print("LSTM SERVER:", json.load(f)["deployment_profile"], "| optimize_on:", json.load(open("/mnt/data/config_lstm_server_opt.json","r")).get("optimize_on"))


## 1) Daten laden, sichten & Feature-Analyse

In [ ]:

# --- Load ---
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]
if "datetime" in df.columns:
    try:
        df["datetime"] = pd.to_datetime(df["datetime"])
        df = df.sort_values("datetime")
        df = df.set_index("datetime")
    except Exception:
        pass

print("Shape:", df.shape)
display(df.head(3))

# --- Basic info ---
nulls = df.isna().mean().sort_values(ascending=False)
print("\nMissing-Rate (Top 10):")
display(nulls.head(10).to_frame("missing_rate"))

# --- Numeric features ---
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cols_to_drop = set([c for c in EXCLUDE_COLS if c in df.columns])
numeric_features = [c for c in num_cols if c not in cols_to_drop and c != TARGET_COL]

print(f"\nNumerische Features (exkl. Zeitspalten & Target): {len(numeric_features)}")
print(numeric_features[:20])

# Visualisierungen
if isinstance(df.index, pd.DatetimeIndex) and TARGET_COL in df.columns:
    plt.figure(figsize=(12,4))
    df[TARGET_COL].plot()
    plt.title("Zielvariable über Zeit")
    plt.xlabel("Zeit")
    plt.ylabel(TARGET_COL)
    plt.show()

if TARGET_COL in df.columns:
    plt.figure(figsize=(6,4))
    df[TARGET_COL].plot(kind="hist", bins=50)
    plt.title("Histogramm Zielvariable")
    plt.xlabel(TARGET_COL)
    plt.show()

corr_cols = [TARGET_COL] + numeric_features[:min(25, len(numeric_features))]
corr = df[corr_cols].corr(numeric_only=True)
plt.figure(figsize=(10,8))
im = plt.imshow(corr.values, aspect='auto')
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(corr_cols)), corr_cols, rotation=90)
plt.yticks(range(len(corr_cols)), corr_cols)
plt.title("Korrelationsmatrix (Subset)")
plt.tight_layout()
plt.show()


## 2) Hilfsfunktionen: Supervised Datensatz & Metriken

In [ ]:

def make_supervised(df, features, target, lags, horizon):
    data = df.copy()
    use_cols = features + [target]
    data = data[use_cols].dropna().copy()
    X_seq, y_seq = [], []
    values = data.values
    for i in range(lags, len(data) - horizon + 1):
        past = values[i-lags:i, :len(features)]
        future = values[i:i+horizon, len(features)]
        X_seq.append(past)
        y_seq.append(future)
    X_3d = np.array(X_seq)
    y_2d = np.array(y_seq)
    X_2d_flat = X_3d.reshape(X_3d.shape[0], -1)
    return X_3d, X_2d_flat, y_2d

def split_train_test(X, y, train_fraction):
    N = len(X)
    split = int(N * train_fraction)
    return X[:split], y[:split], X[split:], y[split:]

def eval_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(len(y_true), -1)
    y_pred = np.asarray(y_pred).reshape(len(y_pred), -1)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = math.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {"mae": mae, "rmse": rmse, "mse": mse, "r2": r2}


## 3) Supervised-Setup & Split

In [ ]:

assert TARGET_COL in df.columns, "Target column not found in data."
X3d, X2d, Y = make_supervised(df, numeric_features, TARGET_COL, LAGS, HORIZON)
print("X3d:", X3d.shape, "| X2d:", X2d.shape, "| Y:", Y.shape)

X3d_tr, Y_tr, X3d_te, Y_te = split_train_test(X3d, Y, TRAIN_FRACTION)
X2d_tr, _,   X2d_te, _   = split_train_test(X2d, Y, TRAIN_FRACTION)
print("Train sizes:", X3d_tr.shape, Y_tr.shape, " | Test sizes:", X3d_te.shape, Y_te.shape)


## 4) Random Forest – Random Search (Edge & Server)

In [ ]:

def sample_from_space(space):
    out = {}
    for k, spec in space.items():
        t = spec.get("type")
        if t == "fixed":
            out[k] = spec["value"]
        elif t == "int":
            out[k] = int(np.random.randint(spec["min"], spec["max"] + 1))
        elif t == "float":
            out[k] = float(np.random.uniform(spec["min"], spec["max"]))
        elif t == "log_float":
            lo, hi = math.log(spec["min"]), math.log(spec["max"])
            out[k] = float(math.exp(np.random.uniform(lo, hi)))
        elif t == "int_choice":
            out[k] = int(np.random.choice(spec["choices"]))
        elif t == "choice":
            out[k] = random.choice(spec["choices"])
        elif t == "int_or_none":
            val = int(np.random.randint(spec["min"], spec["max"] + 1))
            out[k] = None if np.random.rand() < 0.1 else val
        else:
            raise ValueError(f"Unsupported type in search space: {t}")
    return out

with open("/mnt/data/config_rf_edge_opt.json","r",encoding="utf-8") as f:
    rf_edge_cfg = json.load(f)
with open("/mnt/data/config_rf_server_opt.json","r",encoding="utf-8") as f:
    rf_server_cfg = json.load(f)

def rf_random_search(space, trials=40):
    results = []
    best = None
    for t in range(trials):
        params = sample_from_space(space)
        rf = RandomForestRegressor(**{k:v for k,v in params.items() if k in [
            "n_estimators","max_depth","min_samples_split","min_samples_leaf","max_features","bootstrap","random_state","n_jobs"
        ]})
        rf.fit(X2d_tr, Y_tr)
        pred = rf.predict(X2d_te)
        metrics = eval_metrics(Y_te, pred)
        results.append({**params, **metrics})
        if (best is None) or (metrics["rmse"] < best["rmse"]):
            best = {**params, **metrics}
    return pd.DataFrame(results), best

df_rf_edge, best_rf_edge = rf_random_search(rf_edge_cfg["search_space"], trials=30)
df_rf_server, best_rf_server = rf_random_search(rf_server_cfg["search_space"], trials=50)

from caas_jupyter_tools import display_dataframe_to_user
display_dataframe_to_user("RF_Edge_Search_Results", df_rf_edge.sort_values("rmse").head(20))
display_dataframe_to_user("RF_Server_Search_Results", df_rf_server.sort_values("rmse").head(20))

print("Best RF Edge:", best_rf_edge)
print("Best RF Server:", best_rf_server)

rf_edge_final = {**rf_edge_cfg, **{k: best_rf_edge[k] for k in rf_edge_cfg["search_space"].keys() if k in best_rf_edge}}
rf_server_final = {**rf_server_cfg, **{k: best_rf_server[k] for k in rf_server_cfg["search_space"].keys() if k in best_rf_server}}
with open("/mnt/data/rf_edge_final_config.json","w",encoding="utf-8") as f:
    json.dump(rf_edge_final, f, indent=2, ensure_ascii=False)
with open("/mnt/data/rf_server_final_config.json","w",encoding="utf-8") as f:
    json.dump(rf_server_final, f, indent=2, ensure_ascii=False)


## 5) LSTM – Random Search (Edge & Server)

In [ ]:

def build_lstm(input_shape, num_layers=1, initial_units=64, dropout=0.1, horizon=HORIZON):
    model = Sequential()
    units = int(initial_units)
    for i in range(num_layers):
        return_seq = (i < num_layers - 1)
        model.add(LSTM(units, return_sequences=return_seq, input_shape=input_shape if i==0 else None))
        model.add(Dropout(dropout))
        model.add(BatchNormalization())
        units = max(units // 2, 4)
    model.add(Dense(horizon, activation="linear"))
    return model

# Scaling
from sklearn.preprocessing import MinMaxScaler
X3d_tr_ = X3d_tr.reshape(X3d_tr.shape[0], -1)
X3d_te_ = X3d_te.reshape(X3d_te.shape[0], -1)
x_scaler = MinMaxScaler().fit(X3d_tr_)
X3d_tr_scaled = x_scaler.transform(X3d_tr_).reshape(X3d_tr.shape)
X3d_te_scaled = x_scaler.transform(X3d_te_).reshape(X3d_te.shape)
y_scaler = MinMaxScaler().fit(Y_tr)
Y_tr_scaled = y_scaler.transform(Y_tr)
Y_te_scaled = y_scaler.transform(Y_te)

with open("/mnt/data/config_lstm_edge_opt.json","r",encoding="utf-8") as f:
    lstm_edge_cfg = json.load(f)
with open("/mnt/data/config_lstm_server_opt.json","r",encoding="utf-8") as f:
    lstm_server_cfg = json.load(f)

def make_optimizer(p):
    name = str(p.get("optimizer","adam")).lower()
    lr = float(p.get("learning_rate", 1e-3))
    clipnorm = float(p.get("clipnorm", 0.0)) if p.get("clipnorm", 0.0) else None
    kw = { "learning_rate": lr }
    if clipnorm is not None and clipnorm > 0:
        kw["clipnorm"] = clipnorm
    if name == "adam":
        return Adam(**kw)
    elif name == "rmsprop":
        return RMSprop(**kw)
    elif name == "nadam":
        return Nadam(**kw)
    elif name == "adamw":
        # try TF's AdamW; fallback to Adam if not available
        wd = float(p.get("weight_decay", 0.0))
        try:
            return tf.keras.optimizers.AdamW(weight_decay=wd, **kw)
        except Exception:
            return Adam(**kw)
    else:
        return Adam(**kw)

def lstm_random_search(space, trials=20):
    results = []
    best = None
    input_shape = (LAGS, len(numeric_features))
    for t in range(trials):
        p = sample_from_space(space)
        model = build_lstm(
            input_shape=input_shape,
            num_layers=p.get("num_layers", 1),
            initial_units=p.get("initial_units", 64),
            dropout=p.get("dropout", 0.1),
            horizon=HORIZON
        )
        opt = make_optimizer(p)
        model.compile(optimizer=opt, loss=p.get("loss","mse"), metrics=["mae"])

        split = int(len(X3d_tr_scaled) * (1 - VAL_FRACTION))
        X_fit, X_val = X3d_tr_scaled[:split], X3d_tr_scaled[split:]
        y_fit, y_val = Y_tr_scaled[:split], Y_tr_scaled[split:]

        cb = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)]
        history = model.fit(
            X_fit, y_fit,
            validation_data=(X_val, y_val) if len(X_val) > 0 else None,
            epochs=int(p.get("epochs", 40)),
            batch_size=int(p.get("batch_size", 32)),
            verbose=0,
        )

        pred_scaled = model.predict(X3d_te_scaled, verbose=0)
        pred = y_scaler.inverse_transform(pred_scaled)
        metrics = eval_metrics(Y_te, pred)
        results.append({**p, **metrics})
        if (best is None) or (metrics["rmse"] < best["rmse"]):
            best = {**p, **metrics}
    return pd.DataFrame(results), best

df_lstm_edge, best_lstm_edge = lstm_random_search(lstm_edge_cfg["search_space"], trials=20)
df_lstm_server, best_lstm_server = lstm_random_search(lstm_server_cfg["search_space"], trials=30)

from caas_jupyter_tools import display_dataframe_to_user
display_dataframe_to_user("LSTM_Edge_Search_Results", df_lstm_edge.sort_values("rmse").head(20))
display_dataframe_to_user("LSTM_Server_Search_Results", df_lstm_server.sort_values("rmse").head(20))

print("Best LSTM Edge:", best_lstm_edge)
print("Best LSTM Server:", best_lstm_server)

lstm_edge_final = {**lstm_edge_cfg, **{k: best_lstm_edge[k] for k in lstm_edge_cfg["search_space"].keys() if k in best_lstm_edge}}
lstm_server_final = {**lstm_server_cfg, **{k: best_lstm_server[k] for k in lstm_server_cfg["search_space"].keys() if k in best_lstm_server}}
with open("/mnt/data/lstm_edge_final_config.json","w",encoding="utf-8") as f:
    json.dump(lstm_edge_final, f, indent=2, ensure_ascii=False)
with open("/mnt/data/lstm_server_final_config.json","w",encoding="utf-8") as f:
    json.dump(lstm_server_final, f, indent=2, ensure_ascii=False)
